# 02 Cleaning

Apply the full cleaning pipeline to the raw **Retail Store Sales** dataset and export a clean CSV to `data/processed/cleaned_dataset.csv`.

**Cleaning steps (via `scripts/etl_pipeline.basic_clean`):**
1. Drop exact duplicate rows
2. Strip whitespace from all string columns
3. Rename columns to snake_case
4. Parse `transaction_date` → `datetime64`
5. Map `discount_applied` to int (0/1)
6. Impute `total_spent` where possible and drop remaining nulls
7. Fill remaining numeric nulls with column median
8. Standardise categorical label casing (Title Case)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import basic_clean

In [ ]:
RAW_PATH       = PROJECT_ROOT / 'data/raw/retail_store_sales.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df.shape}')
df.head(3)

In [ ]:
clean_df = basic_clean(df)
clean_df.head(3)

## Project-Specific Feature Engineering
In addition to the standard cleaning, we derive key business metrics such as revenue buckets, date parts, and unit pricing.

In [ ]:
from scripts.etl_pipeline import add_derived_features
clean_df = add_derived_features(clean_df)
print(f'Cleaned dataset shape after feature addition: {clean_df.shape}')
clean_df.head(3)

## Post-Cleaning Verification

In [ ]:
print('Remaining nulls per column:')
nulls = clean_df.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'No nulls remaining.')
print()
print('Data types:')
print(clean_df.dtypes)
print()
print('Discount Applied distribution:')
print(clean_df['discount_applied'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## Export Cleaned Dataset

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')
print(f'Final shape: {clean_df.shape}')